In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
from feature_extractor import MinigridFeaturesExtractor
from procedural_level import ProceduralLevel
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv

from minigrid.wrappers import ImgObsWrapper, OneHotPartialObsWrapper

### Training

In [ ]:
# Subprocvecenv recommends 1 env per core
n_envs = os.cpu_count() or 4

# adjust this depending on amt of cores. depending on features_dim it seems ~1k per dim is enough to for a decent model (though this is not scientific at all and just based on my testing)
n_timesteps = 9000


device = "cuda" if torch.cuda.is_available() else "cpu"

# try adjusting features_dim between 128,256,512 but keep in mind it affects training time and timesteps needed
# higher number = more "visual" information on which the PPO model learns on
policy_kwargs = {
    "features_extractor_class": MinigridFeaturesExtractor,
    "features_extractor_kwargs": {"features_dim": 128},
    "normalize_images": False,  # needed for onehot, true would convert values to [0,1]
}


# difficulty 1000 is fine even for starting up a new model training
def make_env() -> ImgObsWrapper:
    """Create a ProceduralLevel environment wrapped with OneHotPartialObsWrapper and ImgObsWrapper."""
    env = ProceduralLevel(render_mode="rgb_array", difficulty=1000, max_steps=120)
    env = OneHotPartialObsWrapper(env)
    return ImgObsWrapper(env)


env = make_vec_env(
    make_env,
    n_envs=n_envs,
    vec_env_cls=SubprocVecEnv,
)

model_level_path = Path("models/one_hot_model_v2.zip")

if model_level_path.exists():
    model = RecurrentPPO.load(model_level_path, env=env)
    print("Model loaded, continuing training")
    model.learn(
        total_timesteps=(n_timesteps * n_envs),
        progress_bar=True,
        reset_num_timesteps=False,
    )
    model.save(model_level_path)
else:
    print("No saved model, Training a new model")
    model = RecurrentPPO(
        "CnnLstmPolicy",
        env,
        n_steps=128,
        batch_size=min(512, (n_envs * 128) // 16),
        policy_kwargs=policy_kwargs,
        verbose=1,
        device=device,
    )
    model.learn(total_timesteps=(n_timesteps * n_envs), progress_bar=False)
    model.save(model_level_path)

### viz evaluation

In [ ]:
def make_env() -> ImgObsWrapper:
    """Create environment with onehot wrapper and img obs wrapper."""
    env = ProceduralLevel(render_mode="human", difficulty=1000, max_steps=100)
    env = OneHotPartialObsWrapper(env)
    return ImgObsWrapper(env)


env = make_vec_env(make_env, n_envs=1, vec_env_cls=DummyVecEnv)

# init model path so that this cell is independent of the cell above, remember to change path!
model_level_path = Path("models/one_hot_model_v2.zip")

model = RecurrentPPO.load(model_level_path, env=env)
obs = env.reset()
lstm_states = None
episode_starts = np.ones((1,), dtype=bool)

while True:
    action, lstm_states = model.predict(
        obs, state=lstm_states, episode_start=episode_starts, deterministic=True
    )

    obs, rewards, dones, info = env.step(action)
    episode_starts = dones

    if dones[0]:
        break

env.close()